# Setup_LoadTest_v3: installer for the v3 load test

Deploys into **this workspace** the 3 notebooks and the 2 pipelines of the package `FabricLoadTest_v3_package.zip`, working out here
every reference (workspace, lakehouse, notebook and pipeline ids). It does not run any test.

## Steps
1. Import this notebook into the target workspace.
2. Create or pick a **lakehouse in the same workspace** and attach it to this notebook as the **default** lakehouse (Explorer, *Add data items*).
3. Upload `FabricLoadTest_v3_package.zip` to `Files/deploy/` of that lakehouse.
4. Adjust the **Parameters** cell (path of the zip and, optionally, the models) and run **Run all**. Start with `DRY_RUN = True` to see the plan.
5. Upload one navigation JSON per model to `Files/PerfScenarios/Queries/` and open `PL_PerfTest_MultiModel_v3`.

## What it does
- Verifies the integrity of the zip (SHA-256 of every file).
- Creates or updates (depending on `OVERWRITE`) the notebooks, bound to the default lakehouse of this notebook.
- Creates or updates the pipelines with the `workspaceId`, the `notebookId` and the child pipeline id **of this workspace**.
- Creates the folders `Files/PerfScenarios/Queries` and `Files/PerfScenarios/logs`.
- Verifies that every id in the deployed items belongs to this workspace and exists, that no placeholder is left unresolved and that the notebooks point to the right lakehouse.

Required permission: contributor role or higher in the workspace.

In [ ]:
# Path of the zip, relative to the default lakehouse of THIS notebook
PACKAGE_PATH = "Files/deploy/FabricLoadTest_v3_package.zip"

# Models to test. None = keep the CHANGE_ME placeholders and edit the `models` parameter of the pipeline afterwards.
# Example:
# MODELS = [
#     {"model_key": "SalesA", "dataset": "Sales A", "workspace": "",
#      "queryfile": "/lakehouse/default/Files/PerfScenarios/Queries/PowerBIPerformanceData_SalesA.json"},
# ]
MODELS = None

OVERWRITE = True   # True: update items that already exist with the same name; False: abort if any exists
DRY_RUN = False    # True: only show the plan, do not create or modify anything

In [ ]:
import base64
import hashlib
import json
import os
import re
import time
import zipfile

import requests

GUID = re.compile(r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}")

ctx = notebookutils.runtime.context
WS_ID = ctx["currentWorkspaceId"]
WS_NAME = ctx.get("currentWorkspaceName")
LH_ID = ctx.get("defaultLakehouseId")
LH_NAME = ctx.get("defaultLakehouseName")
LH_WS = ctx.get("defaultLakehouseWorkspaceId")

if not LH_ID:
    raise RuntimeError("This notebook has no default lakehouse. Attach one in the Explorer panel (Add data items) and run again.")
if LH_WS and LH_WS.lower() != WS_ID.lower():
    raise RuntimeError("The default lakehouse is in another workspace. Use a lakehouse of the same workspace as this notebook.")

zip_file = "/lakehouse/default/" + PACKAGE_PATH.lstrip("/")
if not os.path.exists(zip_file):
    raise FileNotFoundError(f"Package not found: {zip_file}. Upload the zip to that lakehouse or fix PACKAGE_PATH.")
with zipfile.ZipFile(zip_file) as z:
    package = {name: z.read(name) for name in z.namelist() if not name.endswith("/")}

manifest = json.loads(package["manifest.json"])
for item in manifest["items"]:
    if hashlib.sha256(package[item["file"]]).hexdigest() != item["sha256"]:
        raise ValueError(f"{item['file']}: SHA-256 mismatch; the zip is damaged or has been modified.")

print(f"Package: {manifest['package']} {manifest['version']} (built {manifest['built_utc']}), integrity OK")
print(f"Target: workspace '{WS_NAME}' ({WS_ID}), lakehouse '{LH_NAME}' ({LH_ID})")
print(f"Mode: {'DRY RUN' if DRY_RUN else 'DEPLOY'} | overwrite existing items: {OVERWRITE}")

In [ ]:
API = "https://api.fabric.microsoft.com/v1"


def _headers():
    return {"Authorization": f"Bearer {notebookutils.credentials.getToken('pbi')}", "Content-Type": "application/json"}


def _wait(response):
    """Wait for a long-running operation (HTTP 202) and return its result, if any."""
    op = response.headers.get("x-ms-operation-id")
    retry = max(2, int(response.headers.get("Retry-After", "3")))
    deadline = time.time() + 600
    while True:
        time.sleep(retry)
        status = requests.get(f"{API}/operations/{op}", headers=_headers(), timeout=60).json()
        if status.get("status") in ("Failed", "Cancelled"):
            raise RuntimeError(f"Operation {op} ended as {status.get('status')}: {json.dumps(status.get('error'))[:500]}")
        if status.get("status") == "Succeeded":
            break
        if time.time() > deadline:
            raise TimeoutError(f"Operation {op} did not finish within 10 minutes")
    result = requests.get(f"{API}/operations/{op}/result", headers=_headers(), timeout=120)
    return result.json() if result.status_code == 200 and result.content else {}


def api(method, path, body=None, ok=(200, 201, 202, 204)):
    url = path if path.startswith("http") else f"{API}/{path.lstrip('/')}"
    for _ in range(6):
        r = requests.request(method, url, headers=_headers(), data=None if body is None else json.dumps(body), timeout=120)
        if r.status_code == 429:
            time.sleep(int(r.headers.get("Retry-After", "5")))
            continue
        break
    if r.status_code not in ok:
        raise RuntimeError(f"{method} {path} -> HTTP {r.status_code}: {r.text[:600]}")
    if r.status_code == 202:
        return _wait(r)
    return r.json() if r.content else {}


def list_items(kind=None):
    items, url = [], f"workspaces/{WS_ID}/items" + (f"?type={kind}" if kind else "")
    while url:
        page = api("GET", url)
        items += page.get("value", [])
        url = page.get("continuationUri")
    return items


def get_definition(item_id, notebook=False):
    d = api("POST", f"workspaces/{WS_ID}/items/{item_id}/getDefinition" + ("?format=ipynb" if notebook else ""), {})
    return {p["path"]: base64.b64decode(p["payload"]).decode("utf-8") for p in d["definition"]["parts"]}


def _b64(text):
    return base64.b64encode(text.encode("utf-8")).decode("ascii")


# Connection sanity check before touching anything
api("GET", f"workspaces/{WS_ID}")
print("Fabric API reachable with the current identity")

In [ ]:
def clean_models(models):
    """Validate and normalise the list of models (trims spaces, checks keys and format)."""
    if not isinstance(models, list) or not models:
        raise ValueError("MODELS must be a list with at least one model")
    out, seen = [], set()
    for i, m in enumerate(models, start=1):
        missing = {"model_key", "dataset", "queryfile"} - set(m)
        if missing:
            raise ValueError(f"MODELS[{i}]: missing fields {sorted(missing)}")
        entry = {"model_key": str(m["model_key"]).strip(), "dataset": str(m["dataset"]).strip(),
                 "workspace": str(m.get("workspace", "") or "").strip(), "queryfile": str(m["queryfile"]).strip()}
        if not re.fullmatch(r"[A-Za-z0-9_-]+", entry["model_key"]):
            raise ValueError(f"MODELS[{i}].model_key '{entry['model_key']}': use only letters, digits, hyphen or underscore (no spaces)")
        if entry["model_key"] in seen:
            raise ValueError(f"MODELS[{i}].model_key '{entry['model_key']}' is repeated")
        if not entry["dataset"]:
            raise ValueError(f"MODELS[{i}].dataset is empty")
        if not entry["queryfile"].startswith("/lakehouse/default/") or " " in entry["queryfile"]:
            raise ValueError(f"MODELS[{i}].queryfile must start with /lakehouse/default/ and contain no spaces: '{entry['queryfile']}'")
        seen.add(entry["model_key"])
        out.append(entry)
    return out


def notebook_definition(raw):
    nb = json.loads(raw)
    nb.setdefault("metadata", {})["dependencies"] = {"lakehouse": {
        "known_lakehouses": [{"id": LH_ID}], "default_lakehouse": LH_ID,
        "default_lakehouse_name": LH_NAME, "default_lakehouse_workspace_id": WS_ID}}
    return {"format": "ipynb", "parts": [{"path": "notebook-content.ipynb", "payload": _b64(json.dumps(nb, ensure_ascii=False)),
                                          "payloadType": "InlineBase64"}]}


def pipeline_definition(raw, ids):
    text = raw.decode("utf-8").replace("{{WORKSPACE_ID}}", WS_ID)
    for key, value in ids.items():
        text = text.replace("{{" + key + "}}", value)
    if "{{" in text:
        raise ValueError("Unresolved placeholders left in the pipeline: " + ", ".join(sorted(set(re.findall(r"\{\{[^}]+\}\}", text)))))
    doc = json.loads(text)
    if MODELS is not None and "models" in doc["properties"]["parameters"]:
        doc["properties"]["parameters"]["models"]["defaultValue"] = clean_models(MODELS)
    return {"parts": [{"path": "pipeline-content.json", "payload": _b64(json.dumps(doc, ensure_ascii=False)), "payloadType": "InlineBase64"}]}


if MODELS is not None:
    clean_models(MODELS)  # fail early if the models are not valid

existing = {(i["type"], i["displayName"]): i for i in list_items()}
ids = {}       # placeholder -> real id in this workspace
deployed = {}  # item name -> id
plan = []
for item in manifest["items"]:
    key = (item["type"], item["name"])
    if key in existing and not OVERWRITE:
        raise RuntimeError(f"{item['type']} '{item['name']}' already exists and OVERWRITE = False")
    plan.append((item, "update" if key in existing else "create"))

print("Plan:")
for item, action in plan:
    print(f"  {action:<8} {item['type']:<13} {item['name']}")

if DRY_RUN:
    print("\nDRY RUN: nothing was created or modified. Set DRY_RUN = False to deploy.")
else:
    for item, action in plan:
        raw = package[item["file"]]
        notebook = item["type"] == "Notebook"
        definition = notebook_definition(raw) if notebook else pipeline_definition(raw, ids)
        key = (item["type"], item["name"])
        if action == "update":
            item_id = existing[key]["id"]
            api("POST", f"workspaces/{WS_ID}/items/{item_id}/updateDefinition", {"definition": definition})
            api("PATCH", f"workspaces/{WS_ID}/items/{item_id}", {"description": item["description"]})
        else:
            result = api("POST", f"workspaces/{WS_ID}/items", {"displayName": item["name"], "type": item["type"],
                                                               "description": item["description"], "definition": definition})
            item_id = result.get("id") or next((i["id"] for i in list_items(item["type"]) if i["displayName"] == item["name"]), None)
            if not item_id:
                raise RuntimeError(f"Could not get the id of {item['name']} after creating it")
        deployed[item["name"]] = item_id
        prefix = "NOTEBOOK" if notebook else "PIPELINE"
        ids[f"{prefix}:{item['name']}"] = item_id
        print(f"  OK {action:<8} {item['name']} -> {item_id}")

    for folder in manifest["folders"]:
        notebookutils.fs.mkdirs(f"/lakehouse/default/{folder}")
    print("Lakehouse folders ready:", ", ".join(manifest["folders"]))

In [ ]:
if DRY_RUN:
    print("DRY_RUN: verification skipped")
else:
    # Every id found in the deployed items must be this workspace, this lakehouse or an item that exists in this workspace.
    allowed = {WS_ID.lower(), LH_ID.lower()} | {v.lower() for v in deployed.values()}
    present = {i["id"].lower() for i in list_items()}
    problems, rows = [], []

    for item in manifest["items"]:
        notebook = item["type"] == "Notebook"
        item_id = deployed[item["name"]]
        parts = get_definition(item_id, notebook)
        name = "notebook-content.ipynb" if notebook else "pipeline-content.json"
        text = next(v for k, v in parts.items() if k.endswith(name))
        doc = json.loads(text)
        checks = []

        # Notebook ids in cell metadata are not references: scan only the cell sources and the lakehouse binding.
        scanned = text
        if notebook:
            scanned = "\n".join("".join(c["source"]) for c in doc["cells"]) + json.dumps(doc["metadata"].get("dependencies", {}))
        found = {g.lower() for g in GUID.findall(scanned)}

        foreign = found - allowed
        if foreign:
            problems.append(f"{item['name']}: ids that do not belong to this workspace: {sorted(foreign)}")
        missing = {g for g in found if g not in (WS_ID.lower(), LH_ID.lower()) and g not in present}
        if missing:
            problems.append(f"{item['name']}: references to items that do not exist: {sorted(missing)}")
        if "{{" in text:
            problems.append(f"{item['name']}: unresolved placeholders")
        checks.append("no foreign ids, no placeholders")

        if notebook:
            dep = doc["metadata"].get("dependencies", {}).get("lakehouse", {})
            if dep.get("default_lakehouse", "").lower() != LH_ID.lower() or dep.get("default_lakehouse_workspace_id", "").lower() != WS_ID.lower():
                problems.append(f"{item['name']}: wrong default lakehouse: {dep}")
            checks.append("default lakehouse = this one")
        rows.append((item["type"], item["name"], item_id, "; ".join(checks)))

    # The notebook called by the child pipeline must be the one deployed here
    ramp = json.loads(next(v for k, v in get_definition(deployed["PL_PerfTest_Ramp_v3"]).items() if k.endswith("pipeline-content.json")))
    activity = ramp["properties"]["activities"][0]["typeProperties"]["activities"][0]["typeProperties"]
    if activity["notebookId"].lower() != deployed["RunLoadTest_v3"].lower() or activity["workspaceId"].lower() != WS_ID.lower():
        problems.append("PL_PerfTest_Ramp_v3 does not point to the RunLoadTest_v3 of this workspace")
    multi = json.loads(next(v for k, v in get_definition(deployed["PL_PerfTest_MultiModel_v3"]).items() if k.endswith("pipeline-content.json")))
    reference = multi["properties"]["activities"][0]["typeProperties"]["activities"][0]["typeProperties"]["pipeline"]["referenceName"]
    if reference.lower() != deployed["PL_PerfTest_Ramp_v3"].lower():
        problems.append("PL_PerfTest_MultiModel_v3 does not point to the PL_PerfTest_Ramp_v3 of this workspace")

    for row in rows:
        print(f"  {row[0]:<13} {row[1]:<32} {row[2]}  [{row[3]}]")
    if problems:
        print("\nPROBLEMS:")
        for p in problems:
            print("  -", p)
        raise RuntimeError(f"Verification found {len(problems)} problem(s)")
    print("\nVERIFICATION OK: the deployment is self-contained and every reference points to this workspace.")

In [ ]:
base_url = f"https://app.fabric.microsoft.com/groups/{WS_ID}"
if not DRY_RUN:
    print("Pipelines:")
    for name in ("PL_PerfTest_MultiModel_v3", "PL_PerfTest_Ramp_v3"):
        print(f"  {name}: {base_url}/pipelines/{deployed[name]}")
    print("Notebooks:")
    for name in ("RunLoadTest_v3", "RunPerfScenario_v3", "AnalyzeLoadTest_v3"):
        print(f"  {name}: {base_url}/synapsenotebooks/{deployed[name]}")

models = [i["displayName"] for i in list_items("SemanticModel")]
queries_dir = "/lakehouse/default/Files/PerfScenarios/Queries"
files = sorted(os.listdir(queries_dir)) if os.path.isdir(queries_dir) else []
print(f"\nNavigation files in {queries_dir}: {files if files else 'none yet'}")
if MODELS is None:
    print("\nWARNING: the `models` parameter of PL_PerfTest_MultiModel_v3 still holds CHANGE_ME placeholders: set it before running.")
if models:
    suggestion = [{"model_key": re.sub(r"[^A-Za-z0-9_-]+", "", m) or "Model", "dataset": m, "workspace": "",
                   "queryfile": f"{queries_dir}/PowerBIPerformanceData_{re.sub(r'[^A-Za-z0-9_-]+', '', m)}.json"} for m in models]
    print("\nSemantic models in this workspace; suggested value for the `models` parameter (adjust the file names):")
    print(json.dumps(suggestion, ensure_ascii=False))
else:
    print("\nThis workspace has no semantic models yet. Use the `workspace` field of `models` for models that live elsewhere.")

print("""
Next steps:
  1. Record the navigation of each model with Performance Analyzer and upload the JSON to Files/PerfScenarios/Queries/ (one file per model).
  2. Open PL_PerfTest_MultiModel_v3, press Run and fill in `models`. For a first run keep the default ramp_levels = [1, 5, 10].
  3. If you use RLS, fill in `rls_users` (comma-separated UPNs); the identity running the test needs read and build permission on each model.
  4. When it finishes, run AnalyzeLoadTest_v3 to check the ramp and the think times.
Tenant requirement: the XMLA read endpoint must be enabled on the capacity.
""")